# 오차 역전파 Backpropagation

신경망은 순전파를 통해 예측값을 만들고, 손실함수를 통해 정답과의 차이를 계산한다.  
하지만 손실값을 계산하는 것만으로는 학습이 이루어지지 않는다.  
손실을 줄이려면 각 가중치와 편향이 손실에 얼마나 영향을 주는지 알아야 한다.

이때 사용하는 방법이 오차 역전파(backpropagation)이다.

1. 입력에서 출력 방향으로 계산한다. (순전파)
2. 출력에서 계산된 손실을 바탕으로 기울기를 구한다.
3. 그 기울기를 뒤에서 앞으로 전달한다.
4. 각 파라미터가 손실에 얼마나 영향을 주는지 계산한다.

즉, 오차 역전파는 손실을 줄이기 위해 필요한 gradient를 각 층의 파라미터에 대해 계산하는 과정이다.

## 1. 체인룰

역전파의 핵심은 체인룰(chain rule)이다.

신경망의 출력은 여러 연산이 연결된 결과이다.  
따라서 손실을 어떤 파라미터로 미분하려면, 중간 단계의 미분값들을 차례대로 곱해 나가야 한다.

예를 들어 아래와 같은 식을 생각해보자.

- z = 2x + 1
- y = z²

이때 y를 x로 직접 미분하려면,  
중간 변수 z를 거쳐서 변화가 어떻게 전달되는지 봐야 한다.

In [ ]:
import torch
# 입력 값 x를 미분 대상 tensor로 만든다
x = torch.tensor(3.0, requires_grad=True)

# 중간 계산
z = 2 * x + 1
y = z ** 2

print('x : ', x.item())
print('z : ', z.item())
print('y : ', y.item())

# y를 x로 미분한 값을 계산
# dy/dz = 2z
# dz/dx = 2
# dy/dx = dy/dz x dz/dx = 2z x 2
y.backward()
print('dy/dx=', x.grad.item())

# 최종 출력 y가 x에 대해서 얼마나 변하는지는 중간 계산 x를 거치는 변화율까지 모두 곱해서 봐야 한다.
# 값 계산 순서와 미분 계산 순서가 반대이다. => 뒤에서 앞으로 gradient를 전달한다.

x :  3.0
z :  7.0
y :  49.0
dy/dx= 28.0


## 2. 단층 구조에서의 역전파

- 입력 x
- 가중치 w
- 편향 b
- 선형결합 z = wx + b
- 활성화 함수 a = sigmoid(z)
- 손실 L

이 구조에서는 손실 L이 직접 w에 연결되어 있는 것이 아니라, z와 a를 거쳐서 연결되어 있다.

따라서 w에 대한 gradient를 구할 때도 손실이 출력까지 어떻게 전달되었는지를 거꾸로 따라가야 한다.

In [4]:
# 단층 구조를 간단히 수치로 확인
x = torch.tensor(2.0)
w = torch.tensor(1.5, requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)

# 선형 결합
z = w * x + b

# 시그모이드 활성화 함수
a = torch.sigmoid(z)

# 정답을 1이라고 가정하고 간단한 제곱 오차 손실 만들기
target = torch.tensor(1.0)
loss = (a - target) ** 2

print('z : ', z.item())
print('a : ', a.item())
print('loss : ', loss.item())

# loss를 w와 b에 대해서 미분
# 손실은 최종적으로 a에서 계산 되지만, a는 z에 의해 결정 되고, z는 다시 w와 b에 의해 결정 된다.
loss.backward()

print('dL/dw =', w.grad.item())
print('dL/db =', b.grad.item())

z :  3.5
a :  0.970687747001648
loss :  0.0008592081721872091
dL/dw = -0.0033360912930220366
dL/db = -0.0016680456465110183


## 3. 다층 구조에서의 역전파

다층 퍼셉트론에서는 층이 여러 개 연결된다.
예를 들어 아래와 같은 구조를 생각할 수 있다.

- 입력층
- 은닉층 1
- 은닉층 2
- 출력층
- 손실함수

이 경우 손실은 출력층에서 계산되지만, 출력층의 값은 이전 은닉층의 출력에 의존하고, 그 은닉층의 출력은 더 앞쪽 층의 가중치에 의존한다.

즉 층이 많아질수록 체인룰을 통해 gradient를 전달하는 일이 더 중요해진다.

In [ ]:
# 다층 구조를 단순한 계산으로 표현하는 예시
x = torch.tensor(1.0, requires_grad=False)

# 첫 번째 층의 파라미터
w1 = torch.tensor(2.0, requires_grad=True)
b1 = torch.tensor(0.5, requires_grad=True)

# 두 번째 층의 파라미터
w2 = torch.tensor(-1.0, requires_grad=True)
b2 = torch.tensor(1.0, requires_grad=True)

# 순전파
z1 = w1 * x + b1
a1 = torch.relu(z1)         # 은닉층 활성화 함수로 ReLU 사용
z2 = w2 * a1 + b2           # 출력층 선형 결합
target = torch.tensor(0.0)

# 간단한 회귀 손실
loss = (z2 - target) ** 2

print('z1:', z1.item())
print('a1', a1.item())
print('z2:', z2.item())
print('loss:', loss.item())

# 역전파 - 기울기 계산
loss.backward()

# dL/dw2,dL/db2 값이 음수라는 것은 w2, b2를 증가시키면 손실이 줄어든다는 뜻
# dL/dw1,dL/db1 값이 양수라는 것은 w1, b2를 감소시키면 손실이 줄어든다는 뜻
print('dL/dw1=', w1.grad.item())
print('dL/db1=', b1.grad.item())
print('dL/dw2=', w2.grad.item())
print('dL/db2=', b2.grad.item())

z1: 2.5
a1 2.5
z2: -1.5
loss: 2.25
dL/dw1= 3.0
dL/db1= 3.0
dL/dw2= -7.5
dL/db2= -3.0


## 4. 활성화 함수의 도함수

도함수는 어떤 함수를 미분해서 얻은 결과이다.
즉, 원래 함수의 기울기를 나타내는 새로운 함수라고 볼 수 있다.

1. Sigmoid
- 출력이 0과 1 사이
- 원래 식: `sigmoid(x) = 1 / (1 + e^(-x))`
- 도함수: `sigmoid(x) * (1 - sigmoid(x))`
- 가운데 부근에서는 도함수 값이 비교적 크고, 양끝으로 갈수록 도함수 값이 작아진다.

2. ReLU
- 원래 식: `x > 0` 이면 f(x) = x, `x <= 0` 이면 f(x) = 0
- 도함수:`x > 0` 이면 1, `x <= 0` 이면 0

즉 역전파에서는 각 층을 지날 때마다 이런 도함수 값이 곱해지면서 gradient가 전달된다.

이 말은 곧, 도함수 값이 아주 작으면 gradient도 점점 작아질 수 있고, 도함수 값이 0이면 그 지점에서는 gradient 전달이 끊길 수도 있다는 뜻이다.

In [ ]:
x = torch.linspace(-5, 5, 11)

# 시그모이드 함수의 출력 값
sigmoid_x = torch.sigmoid(x)

# sigmoid의 도함수
sigmoid_grad = sigmoid_x * (1 - sigmoid_x)

# relu 함수의 출력 값
relu_x = torch.relu(x)

# relu의 도함수
relu_grad = torch.where(x > 0, torch.tensor(1.0), torch.tensor(0.0))

print('x:', x)
print('sigmoid_x:', sigmoid_x)
# sigmoid의 도함수는 가운데(0 근처)에서 가장 크고, 양 끝으로 갈수록 매우 작아진다.
# => 입력 값이 너무 크거나 너무 작아지면 gradient가 작아질 수 있다. (학습이 잘 안될 수 있음)
print('sigmoid_grad:', sigmoid_grad)
print('relu_x:', relu_x)
# ReLU의 도함수는 0이하의 구간에서 0이므로 gradient가 전달 되지 않는다. (학습이 잘 안될 수 있음)
print('relu_grad:', relu_grad)

x: tensor([-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.])
sigmoid_x: tensor([0.0067, 0.0180, 0.0474, 0.1192, 0.2689, 0.5000, 0.7311, 0.8808, 0.9526,
        0.9820, 0.9933])
sigmoid_grad: tensor([0.0066, 0.0177, 0.0452, 0.1050, 0.1966, 0.2500, 0.1966, 0.1050, 0.0452,
        0.0177, 0.0066])
relu_x: tensor([0., 0., 0., 0., 0., 0., 1., 2., 3., 4., 5.])
relu_grad: tensor([0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1.])


## 5. PyTorch에서 역전파가 자동으로 이루어지는 방식

실제 구현에서는 PyTorch의 autograd가 이 과정을 자동으로 처리해준다.
즉 우리가 각 층의 미분 공식을 일일이 손으로 코딩하지 않아도, 연산 그래프를 기반으로 gradient를 자동 계산할 수 있다.

In [8]:
# 입력 값 (feature 2개)
x = torch.tensor([[1.0, 2.0]], requires_grad=False) 
# 가중치 
W = torch.tensor([[0.5], [1.5]], requires_grad=True)
# 편향
b = torch.tensor([0.2], requires_grad=True)

# 순전파
z = x @ W + b                   # 선형 결합
a = torch.sigmoid(z)            # 활성화 함수
target = torch.tensor([[1.0]])  # 정답
loss = (a - target) ** 2        # 손실 값

# 역전파
loss.backward()

print('W.grad')
print(W.grad)
print('b.grad')
print(b.grad)



W.grad
tensor([[-0.0011],
        [-0.0023]])
b.grad
tensor([-0.0011])
